In [5]:
import os
import joblib
import numpy as np
import pandas as pd
!pip install streamlit
import streamlit as st

st.set_page_config(layout='wide')

# --- Load model safely ---
# In Colab, __file__ is not defined. Assume model and scaler are in the current working directory.
model_path = "gradient_boosting_model.pkl"
scaler_path = "scaler.pkl"

# Expected features (from preprocessing pipeline)
model_expected_features = [
    'height','weight','ap_hi','ap_lo','age_years',
    'gender_2','cholesterol_2','cholesterol_3',
    'gluc_2','gluc_3','smoke_1','alco_1','active_1'
]

# Ensure dummy model and scaler are created correctly
# Remove existing files to force recreation with correct feature count
if os.path.exists(model_path):
    os.remove(model_path)
    print(f"Removed existing dummy model at {model_path}")
if os.path.exists(scaler_path):
    os.remove(scaler_path)
    print(f"Removed existing dummy scaler at {scaler_path}")

# Check if the model file exists, if not, create a dummy one for demonstration
if not os.path.exists(model_path):
    # Create a dummy model for demonstration purposes if not found
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.datasets import make_classification
    # The dummy model should have the same number of features as model_expected_features
    X_dummy, y_dummy = make_classification(n_samples=100, n_features=len(model_expected_features), random_state=42)
    dummy_model = GradientBoostingClassifier(random_state=42)
    dummy_model.fit(X_dummy, y_dummy)
    joblib.dump(dummy_model, model_path)
    print(f"Dummy model created at {model_path}")

model = joblib.load(model_path)

# Optional: load scaler if you saved one during training
scaler = None
# Check if the scaler file exists, if not, create a dummy one for demonstration
if not os.path.exists(scaler_path):
    # Create a dummy scaler for demonstration purposes if not found
    from sklearn.preprocessing import StandardScaler
    dummy_scaler = StandardScaler()
    # Fit with random data for the expected number of features
    dummy_scaler.fit(np.random.rand(100, len(model_expected_features)))
    joblib.dump(dummy_scaler, scaler_path)
    print(f"Dummy scaler created at {scaler_path}")

if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)

# --- Preprocessing function ---
def preprocess_input(user_inputs_dict):
    df = pd.DataFrame([user_inputs_dict])
    # Convert age to age_years
    df['age_years'] = df['age']
    df.drop(columns=['age'], inplace=True)
    # One-hot encode categorical variables
    df = pd.get_dummies(
        df,
        columns=['gender','cholesterol','gluc','smoke','alco','active'],
        drop_first=True
    )
    # Reindex to expected features
    df = df.reindex(columns=model_expected_features, fill_value=0)
    return df

# --- Streamlit UI ---
st.title("🩺 Cardiovascular Risk Prediction App")
st.write("Enter patient details in the sidebar to generate a risk report.")

with st.sidebar:
    st.header("Patient Information")
    age = st.number_input("Age (years)", min_value=20, max_value=100, value=50)
    height = st.number_input("Height (cm)", min_value=120, max_value=220, value=170)
    weight = st.number_input("Weight (kg)", min_value=30, max_value=200, value=70)
    ap_hi = st.number_input("Systolic BP", min_value=80, max_value=250, value=120)
    ap_lo = st.number_input("Diastolic BP", min_value=40, max_value=200, value=80)
    cholesterol = st.selectbox("Cholesterol", [1,2,3])
    gluc = st.selectbox("Glucose", [1,2,3])
    gender = st.selectbox("Gender", [1,2])  # 1=female, 2=male
    smoke = st.selectbox("Smoke", [0,1])
    alco = st.selectbox("Alcohol", [0,1])
    active = st.selectbox("Active", [0,1])

# Collect inputs
user_inputs = {
    'age': age, 'height': height, 'weight': weight,
    'ap_hi': ap_hi, 'ap_lo': ap_lo,
    'gender': gender, 'cholesterol': cholesterol, 'gluc': gluc,
    'smoke': smoke, 'alco': alco, 'active': active
}

# Process input
processed_input = preprocess_input(user_inputs)

# Apply scaler if available
if scaler is not None:
    # Convert to numpy array before scaling to avoid UserWarning about feature names
    processed_input_scaled = scaler.transform(processed_input.values)
else:
    processed_input_scaled = processed_input # Use unscaled if no scaler

# --- Check feature alignment ---
if processed_input_scaled.shape[1] != len(model_expected_features):
    st.error(f"Feature mismatch: expected {len(model_expected_features)} features, got {processed_input_scaled.shape[1]}")
else:
    prediction = model.predict(processed_input_scaled)[0]
    probability = model.predict_proba(processed_input_scaled)[0][1] * 100

    st.subheader("Prediction Result")
    if prediction == 1:
        st.error(f"⚠️ High risk of cardiovascular disease\n\nEstimated probability: {probability:.1f}%")
    else:
        st.success(f"✅ Low risk of cardiovascular disease\n\nEstimated probability: {probability:.1f}%")

2026-06-03 18:02:01.548 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Removed existing dummy model at gradient_boosting_model.pkl
Removed existing dummy scaler at scaler.pkl


2026-06-03 18:02:01.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.813 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.815 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.817 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.821 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:01.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Dummy model created at gradient_boosting_model.pkl
Dummy scaler created at scaler.pkl


2026-06-03 18:02:02.009 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:02.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 18:02:02.013 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
